# Lecture 4c — Serving with Ray Serve (replicas, canary routing, autoscaling)
**AI Operations · Module 4 · CPU only · core path ≈ 20 min in class · full notebook ≈ 60 min (executes in ≈ 6 min)**

**You will:** (1) turn a Python class into a scalable **deployment**, (2) compose a **router → models** graph with a weighted **canary split**, (3) scale replicas manually and automatically, (4) watch Ray **self-heal** a killed replica, (5) run a promote/rollback drill, (6) build a new composed deployment (ensemble).

Ray Serve runs *inside* a Python program (or a Ray cluster) — no containers required. Replicas are Ray **actors** (you met them in Module 2), reserved with *fractional* CPUs so a laptop can host many.

In [ ]:
import sys, subprocess, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ray[serve]", "scikit-learn", "pandas", "joblib", "requests", "matplotlib", "fastapi"], check=False)
os.makedirs("data", exist_ok=True)

In [ ]:
%%writefile train_models.py
"""
train_models.py  --  AIOps Module 4 (Model Serving)
Trains TWO versions of a customer-churn classifier on a synthetic, deterministic dataset.
  v1 = LogisticRegression (fast, simple baseline)      -> data/model_v1.joblib
  v2 = RandomForest       (slower, usually more accurate) -> data/model_v2.joblib
Two versions let us practise canary releases, blue-green switches and rollback later.
CPU only; runs in a few seconds.
"""
import json, argparse, numpy as np, pandas as pd, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

FEATURES = ["tenure_months", "monthly_charge", "support_calls", "late_payments",
            "data_usage_gb", "contract_months", "num_products", "age"]

def make_data(n=6000, seed=7):
    r = np.random.default_rng(seed)
    df = pd.DataFrame({
        "tenure_months": r.integers(1, 72, n),
        "monthly_charge": r.normal(65, 20, n).clip(15, 150).round(2),
        "support_calls": r.poisson(2, n),
        "late_payments": r.poisson(0.7, n),
        "data_usage_gb": r.gamma(3, 8, n).round(2),
        "contract_months": r.choice([1, 12, 24], n, p=[.5, .3, .2]),
        "num_products": r.integers(1, 6, n),
        "age": r.integers(18, 80, n),
    })
    z = (-2.0 - 0.02*df.tenure_months + 0.01*df.monthly_charge + 0.15*df.support_calls
         + 0.2*df.late_payments - 0.01*df.contract_months - 0.1*df.num_products
         + 2.2*((df.support_calls >= 4) & (df.contract_months == 1))      # non-linear interactions:
         + 1.8*((df.tenure_months < 12) & (df.monthly_charge > 80))       # a linear model cannot
         + 1.5*((df.late_payments >= 2) & (df.num_products <= 2))         # capture these well
         - 1.6*((df.contract_months == 24) & (df.tenure_months > 36))
         + r.normal(0, .5, n))
    df["churn"] = (1/(1+np.exp(-z)) > r.uniform(size=n)).astype(int)
    return df

def main(out="data"):
    df = make_data()
    X, y = df[FEATURES], df["churn"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=0, stratify=y)
    models = {
        "v1": make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
        "v2": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=0, n_jobs=1),
    }
    report = {}
    for ver, m in models.items():
        m.fit(Xtr, ytr)
        acc = accuracy_score(yte, m.predict(Xte))
        joblib.dump({"model": m, "feature_columns": FEATURES, "version": ver}, f"{out}/model_{ver}.joblib")
        report[ver] = round(acc, 4)
        print(f"{ver}: test accuracy = {acc:.4f}")
    df.to_csv(f"{out}/churn.csv", index=False)
    Xte.head(200).to_csv(f"{out}/sample_requests.csv", index=False)
    json.dump({"rows": Xte.head(3).to_dict(orient="records")}, open(f"{out}/sample_payload.json", "w"), indent=2)
    json.dump(report, open(f"{out}/metrics.json", "w"))

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--out", default="data"); main(ap.parse_args().out)

In [ ]:
%%writefile ray_serve_app.py
"""
ray_serve_app.py -- AIOps Module 4, Topic 3: Ray Serve (CPU only)
Architecture:   client --HTTP--> Router (ingress, canary split) --> ChurnModel[v1] x N replicas
                                                               +-> ChurnModel[v2] x M replicas
Run:   python ray_serve_app.py          (starts Ray + Serve, then blocks)  or  serve run ray_serve_app:app
Env:   CANARY_WEIGHT (0.0 - 1.0, share of traffic sent to v2; default 0.2)
"""
import os, random, time, joblib, pandas as pd
from typing import List, Dict
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from ray import serve
from ray.serve.handle import DeploymentHandle

FEATURES = ["tenure_months", "monthly_charge", "support_calls", "late_payments",
            "data_usage_gb", "contract_months", "num_products", "age"]

@serve.deployment(
    num_replicas=2, ray_actor_options={"num_cpus": 0.25},          # 0.25 CPU per replica: fits a small laptop
    max_ongoing_requests=16,                                       # back-pressure per replica
    health_check_period_s=5,
    logging_config={"log_level": "WARNING"})
class ChurnModel:
    def __init__(self, path: str, extra_work_ms: float = 0.0):
        # extra_work_ms simulates a LARGER model (e.g. a deep net). It is a blocking sleep, not a CPU spin,
        # so replica-count effects stay visible even on a 2-core laptop. Keep it 0 for real work.
        self.extra = extra_work_ms / 1000.0
        b = joblib.load(path)                                      # model loaded once per replica
        self.model, self.version = b["model"], b["version"]
        self.replica = serve.get_replica_context().replica_id.unique_id
    async def __call__(self, rows: List[Dict]):
        X = pd.DataFrame(rows)[FEATURES]
        p = self.model.predict_proba(X)[:, 1]
        if self.extra: time.sleep(self.extra)
        return {"predictions": [int(x >= .5) for x in p], "probabilities": [round(float(x), 4) for x in p],
                "model_version": self.version, "served_by": self.replica}
    async def check_health(self):                                        # called by Serve; raise => replica restarted
        if self.model is None: raise RuntimeError("model missing")

class Customer(BaseModel):
    tenure_months: int; monthly_charge: float; support_calls: int; late_payments: int
    data_usage_gb: float; contract_months: int; num_products: int; age: int
class Req(BaseModel):
    instances: List[Customer]

api = FastAPI()

@serve.deployment(num_replicas=1, ray_actor_options={"num_cpus": 0.1}, logging_config={"log_level": "WARNING"})
@serve.ingress(api)
class Router:
    def __init__(self, v1: DeploymentHandle, v2: DeploymentHandle, canary_weight: float):
        self.h = {"v1": v1, "v2": v2}
        self.w = canary_weight
    @api.get("/healthz")
    def healthz(self): return {"status": "ok", "canary_weight": self.w}
    @api.post("/v1/models/churn:predict")
    async def predict(self, req: Req):
        arm = "v2" if random.random() < self.w else "v1"            # weighted traffic split = canary
        try:
            return await self.h[arm].remote([c.model_dump() for c in req.instances])
        except Exception as e:
            raise HTTPException(500, f"{arm} failed: {type(e).__name__}")

def build_app(canary_weight: float = 0.2, v1_replicas: int = 2, v2_replicas: int = 2, extra_work_ms: float = 0.0):
    """Compose the application graph: Router -> (ChurnModel v1, ChurnModel v2)."""
    v1 = ChurnModel.options(name="ChurnModelV1", num_replicas=v1_replicas).bind("data/model_v1.joblib", extra_work_ms)
    v2 = ChurnModel.options(name="ChurnModelV2", num_replicas=v2_replicas).bind("data/model_v2.joblib", extra_work_ms)
    return Router.bind(v1, v2, canary_weight)

app = build_app(float(os.getenv("CANARY_WEIGHT", "0.2")))

if __name__ == "__main__":
    import ray
    ray.init(num_cpus=int(os.getenv("RAY_CPUS", "2")), include_dashboard=False, ignore_reinit_error=True)
    serve.start(http_options={"host": "0.0.0.0", "port": 8100})
    serve.run(app, route_prefix="/")
    print("Serve is up on :8100"); 
    while True: time.sleep(3600)

In [ ]:
%%writefile load_test.py
"""
load_test.py -- generic closed-loop load generator for any model endpoint (REST / MLflow / Ray / K8s).
  python load_test.py --url http://localhost:8000/v1/models/churn:predict --format rest
  python load_test.py --url http://localhost:5001/invocations --format mlflow --concurrency 1 4 8
Reports p50/p95/p99 latency, throughput, errors and which replicas / model versions answered.
"""
import argparse, json, time, statistics, collections, concurrent.futures as cf
import pandas as pd, requests

def payload(rows, fmt, batch):
    recs = rows[:batch]
    if fmt == "rest":   return {"instances": recs}
    if fmt == "mlflow": return {"dataframe_records": recs}
    raise ValueError(fmt)

def one(sess, url, body):
    t0 = time.perf_counter()
    try:
        r = sess.post(url, json=body, timeout=30)
        ok = r.status_code == 200
        j = r.json() if ok else {}
        who = j.get("served_by", "n/a") if isinstance(j, dict) else "n/a"
        ver = j.get("model_version", "n/a") if isinstance(j, dict) else "n/a"
        if ver == "n/a" and isinstance(j, dict) and j.get("predictions") and isinstance(j["predictions"][0], dict):
            ver = j["predictions"][0].get("model_version", "n/a")      # MLflow pyfunc output
    except Exception:
        ok, who, ver = False, "n/a", "n/a"
    return (time.perf_counter() - t0) * 1000, ok, who, ver

def run(url, fmt, conc, duration, rows, batch):
    body = payload(rows, fmt, batch)
    lat, errs, who, ver = [], 0, collections.Counter(), collections.Counter()
    end = time.time() + duration
    def worker():
        s = requests.Session(); out = []
        while time.time() < end:
            out.append(one(s, url, body))
        return out
    with cf.ThreadPoolExecutor(conc) as ex:
        for res in [f.result() for f in [ex.submit(worker) for _ in range(conc)]]:
            for ms, ok, w, v in res:
                lat.append(ms); errs += (not ok); who[w] += 1; ver[v] += 1
    lat.sort(); n = len(lat); q = lambda p: lat[min(n-1, int(p*n))]
    return dict(concurrency=conc, requests=n, rps=round(n/duration, 1), p50_ms=round(q(.5), 1),
                p95_ms=round(q(.95), 1), p99_ms=round(q(.99), 1), errors=errs,
                replicas=len(who), versions=dict(ver))

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", required=True); ap.add_argument("--format", default="rest")
    ap.add_argument("--concurrency", type=int, nargs="+", default=[1, 4, 8])
    ap.add_argument("--duration", type=float, default=5); ap.add_argument("--batch", type=int, default=1)
    ap.add_argument("--data", default="data/sample_requests.csv"); ap.add_argument("--out")
    a = ap.parse_args()
    rows = pd.read_csv(a.data).to_dict(orient="records")
    results = [run(a.url, a.format, c, a.duration, rows, a.batch) for c in a.concurrency]
    print(pd.DataFrame(results).to_string(index=False))
    if a.out: json.dump(results, open(a.out, "w"), indent=2)

In [ ]:
%%writefile m4util.py
"""m4util.py -- tiny helpers shared by the Module 4 notebooks (start / wait for / stop server processes)."""
import subprocess, time, os, signal, sys, requests

_procs = []

def start(cmd, log="server.log", env=None):
    """Start a server as a child process; stdout/stderr go to a log file (read it when things break)."""
    e = {**os.environ, **(env or {})}
    f = open(log, "w")
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=f, stderr=subprocess.STDOUT, env=e,
                         preexec_fn=os.setsid)
    _procs.append(p)
    return p

def wait_ready(url, timeout=90):
    """Poll a URL until it answers 200 (readiness probe, exactly what Kubernetes does)."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return round(time.time() - t0, 1)
        except Exception:
            pass
        time.sleep(0.5)
    raise TimeoutError(f"{url} not ready after {timeout}s -- check the log file")

def stop(p, timeout=10):
    """Stop one server: SIGTERM the whole process group (so uvicorn/gunicorn workers die too),
    then WAIT until the group is really gone -- otherwise a restart can 'succeed' against the old process."""
    try:
        pgid = os.getpgid(p.pid)
    except Exception:
        return
    try:
        os.killpg(pgid, signal.SIGTERM)
        t0 = time.time()
        while time.time() - t0 < timeout:
            try: os.killpg(pgid, 0)            # signal 0 = "does the group still exist?"
            except ProcessLookupError: break
            time.sleep(0.2)
        else:
            os.killpg(pgid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    try: p.wait(timeout=2)
    except Exception: pass

def stop_all():
    for p in _procs: stop(p)
    _procs.clear()

In [ ]:
!python train_models.py

## 1 · Start Ray and deploy the application

In [ ]:
import ray, time, requests, json, collections, pandas as pd, numpy as np
from ray import serve
ray.init(num_cpus=2, include_dashboard=False, ignore_reinit_error=True, logging_level="ERROR", log_to_driver=False)
serve.start(http_options={"host": "127.0.0.1", "port": 8100}, logging_config={"log_level": "WARNING"})
print(ray.cluster_resources())

We told Ray it has **2 CPUs**. Each model replica asks for `num_cpus=0.25` and the router `0.1`, so the default app needs 2×0.25 + 2×0.25 + 0.1 = **1.1 CPU**. Open `ray_serve_app.py` above and find: `@serve.deployment(num_replicas=…, ray_actor_options=…, max_ongoing_requests=…)` — that decorator is the whole scaling interface.

In [ ]:
import ray_serve_app
handle = serve.run(ray_serve_app.build_app(canary_weight=0.2), route_prefix="/", name="default")
BASE = "http://127.0.0.1:8100"
print(requests.get(f"{BASE}/healthz").json())
data = pd.read_csv("data/sample_requests.csv"); recs = data.to_dict(orient="records")
r = requests.post(f"{BASE}/v1/models/churn:predict", json={"instances": recs[:2]})
print(r.status_code, r.json())

In [ ]:
st = serve.status().applications["default"]
pd.DataFrame([{"deployment": n, "status": str(d.status).split(".")[-1], "replicas": dict(d.replica_states)} for n, d in st.deployments.items()])

## 2 · Canary: measure the traffic split

In [ ]:
def split(n=300, url=f"{BASE}/v1/models/churn:predict"):
    sess = requests.Session(); ver = collections.Counter(); who = collections.defaultdict(set); err = 0
    for i in range(n):
        r = sess.post(url, json={"instances": [recs[i % len(recs)]]})
        if r.status_code != 200: err += 1; continue
        j = r.json(); ver[j["model_version"]] += 1; who[j["model_version"]].add(j["served_by"])
    return {"errors": err, **{v: f"{k} ({k/n:.0%}) from {len(who[v])} replicas" for v, k in ver.items()}}
split()

With `canary_weight=0.2` roughly 20 % of requests reach **v2**, and each version is served by **two different replicas** (`served_by` differs). 300 samples gives ±~4 points of statistical noise — one reason real canaries run until they've seen enough traffic to be meaningful.

## 3 · Manual scaling: does adding replicas add throughput?

A 2-core sandbox cannot show replica scaling with the *real* (tiny) model: the HTTP proxy, router and load generator saturate the CPUs first. So here we make the model **heavier on purpose**: `extra_work_ms=80` adds an 80 ms blocking step per request inside each replica (a stand-in for a bigger network; it sleeps, so it costs no CPU). One replica can then do at most ≈ 1000/80 ≈ 12 requests/s.

In [ ]:
import load_test
URL = f"{BASE}/v1/models/churn:predict"
out = []
for n in [1, 2, 4]:
    serve.run(ray_serve_app.build_app(canary_weight=1.0, v1_replicas=1, v2_replicas=n, extra_work_ms=80), route_prefix="/", name="default")
    time.sleep(3)
    load_test.run(URL, "rest", 4, 2, recs, 1)                         # warm-up
    r = load_test.run(URL, "rest", 8, 6, recs, 1); r["v2_replicas"] = n; out.append(r)
scal = pd.DataFrame(out)[["v2_replicas", "rps", "p50_ms", "p95_ms", "replicas", "errors"]]
scal["speed-up"] = (scal.rps / scal.rps.iloc[0]).round(2)
scal

**Read it.** Throughput grows (about 1.7× with two replicas on our run) while the *replicas* are the bottleneck, and latency falls because requests queue less; the curve bends as other stages take over. It will stop growing when another stage — here the single router and HTTP proxy, a few dozen requests/s in this sandbox — becomes the limit. **Scale the bottleneck, not just the model.** On a real multi-node cluster (and minikube in Lecture 4d) the same knob adds real cores.

## 4 · Autoscaling from load

In [ ]:
auto = ray_serve_app.ChurnModel.options(
    name="ChurnAutoscale", num_replicas=None,
    autoscaling_config=dict(min_replicas=1, max_replicas=4, target_ongoing_requests=1,
                            upscale_delay_s=2, downscale_delay_s=10, metrics_interval_s=1, look_back_period_s=3))
small_v2 = ray_serve_app.ChurnModel.options(name="ChurnModelV2", num_replicas=1)
serve.run(ray_serve_app.Router.bind(auto.bind("data/model_v1.joblib", 0.0), small_v2.bind("data/model_v2.joblib", 0.0), 0.0),
          route_prefix="/", name="default")
def n_replicas(name="ChurnAutoscale"):
    return dict(serve.status().applications["default"].deployments[name].replica_states)
print("idle:", n_replicas())

In [ ]:
import threading
timeline = []
stop_flag = False
def poll():
    t0 = time.time()
    while not stop_flag:
        timeline.append((round(time.time() - t0, 1), sum(n_replicas().values()))); time.sleep(1)
th = threading.Thread(target=poll); th.start()
load_test.run(URL, "rest", 12, 25, recs, 1)           # 12 concurrent clients for 25 s
time.sleep(1); idle_start = len(timeline)
time.sleep(18)                                         # idle -> downscale after downscale_delay_s
stop_flag = True; th.join()
tl = pd.DataFrame(timeline, columns=["t_s", "replicas"])
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 2.8)); plt.step(tl.t_s, tl.replicas, where="post"); plt.axvline(tl.t_s[idle_start], ls="--", c="gray")
plt.xlabel("seconds"); plt.ylabel("replicas"); plt.title("Autoscaling: load for 25 s, then idle (dashed)"); plt.show()
print("max replicas reached:", tl.replicas.max(), "| final:", tl.replicas.iloc[-1])

The controller observes **ongoing requests per replica** and scales toward `target_ongoing_requests`. `upscale_delay_s` / `downscale_delay_s` damp flapping: scale **up fast, down slowly** — the same philosophy as the Kubernetes HPA stabilization window you will see in Lecture 4d.

## 5 · Self-healing: kill a replica

In [ ]:
serve.delete("default"); time.sleep(3)        # remove the autoscaled app cleanly before redeploying
serve.run(ray_serve_app.build_app(canary_weight=0.0, v1_replicas=2, v2_replicas=1), route_prefix="/", name="default")
time.sleep(3)
names = [a for a in ray.util.list_named_actors(all_namespaces=True) if "ChurnModelV1" in a["name"] and "REPLICA" in a["name"]]
print("v1 replica actors:", len(names))

# kill one of the v1 replicas and see how quickly the controller replaces it
victim = ray.get_actor(names[0]["name"], namespace=names[0]["namespace"]); ray.kill(victim, no_restart=True)
ok = err = 0; t0 = time.time(); seen = []; last = 0
while time.time() - t0 < 12:
    try:
        r = requests.post(URL, json={"instances": recs[:1]}, timeout=5); ok += r.status_code == 200; err += r.status_code != 200
    except requests.RequestException:
        err += 1
    if time.time() - last >= 1:                                     # sample the replica count once per second
        seen.append((round(time.time() - t0, 1), sum(n_replicas("ChurnModelV1").values()))); last = time.time()
    time.sleep(0.1)
print(f"during/after the kill: {ok} OK, {err} errors; replica count samples (t, n):", seen[::3])
print('(the controller replaces a dead replica within about a second, so the count rarely dips visibly)')
print("final:", n_replicas("ChurnModelV1"))

The controller notices the dead actor and starts a replacement; the router skips unavailable replicas, so clients see (almost) no errors. This is the *same promise* Kubernetes gives for pods — implemented one layer lower.

## 6 · Promote and rollback drill

In [ ]:
drill = []
for label, w in [("canary 20%", 0.2), ("canary 50%", 0.5), ("PROMOTE 100%", 1.0), ("ROLLBACK 0%", 0.0)]:
    serve.run(ray_serve_app.build_app(canary_weight=w), route_prefix="/", name="default"); time.sleep(2)
    s = split(150); drill.append({"stage": label, "weight": w, **{k: v for k, v in s.items()}})
pd.DataFrame(drill).fillna("-")

Changing the router's argument redeploys **only the router** — model replicas keep their loaded weights (no cold start), so a rollback is as quick as a promote.

## 7 · YOUR TURN

**Task A — an ensemble deployment.** Create a deployment `Ensemble` that receives the same rows, calls **v1 and v2 in parallel** (`asyncio.gather` over two `DeploymentHandle`s) and returns the **average probability** and the label at 0.5. Deploy it on `/` and verify with one request that `probabilities` lies between the individual models' values.

**Task B — pick an arm under an SLO.** Using `canary_weight` 0.0 and 1.0 (2 replicas each) and 4 concurrent clients, measure p95 latency for v1-only and v2-only. Our SLO is **p95 ≤ 1.5 × the v1-only p95** (absolute numbers depend on your hardware). Does v2 meet it, and what does that mean for a 100 % rollout of v2?

In [ ]:
# Task A — your code here

**Solution A**

In [ ]:
import asyncio
from fastapi import FastAPI
api2 = FastAPI()
@serve.deployment(num_replicas=1, ray_actor_options={"num_cpus": 0.1})
@serve.ingress(api2)
class Ensemble:
    def __init__(self, h1, h2): self.h1, self.h2 = h1, h2
    @api2.post("/v1/models/churn:predict")
    async def predict(self, req: ray_serve_app.Req):
        rows = [c.model_dump() for c in req.instances]
        a, b = await asyncio.gather(self.h1.remote(rows), self.h2.remote(rows))      # parallel fan-out
        p = [round((x + y) / 2, 4) for x, y in zip(a["probabilities"], b["probabilities"])]
        return {"probabilities": p, "predictions": [int(x >= .5) for x in p], "members": [a["model_version"], b["model_version"]]}
    @api2.get("/healthz")
    def h(self): return {"status": "ok"}

v1 = ray_serve_app.ChurnModel.options(name="ChurnModelV1", num_replicas=1).bind("data/model_v1.joblib", 0.0)
v2 = ray_serve_app.ChurnModel.options(name="ChurnModelV2", num_replicas=1).bind("data/model_v2.joblib", 0.0)
serve.run(Ensemble.bind(v1, v2), route_prefix="/", name="default"); time.sleep(2)
e = requests.post(URL, json={"instances": recs[:1]}).json()
serve.run(ray_serve_app.build_app(0.0), route_prefix="/", name="default"); time.sleep(2)
a = requests.post(URL, json={"instances": recs[:1]}).json()["probabilities"][0]
serve.run(ray_serve_app.build_app(1.0), route_prefix="/", name="default"); time.sleep(2)
b = requests.post(URL, json={"instances": recs[:1]}).json()["probabilities"][0]
print("ensemble:", e, "| v1:", a, "| v2:", b, "| between?", min(a, b) - 1e-3 <= e["probabilities"][0] <= max(a, b) + 1e-3)

In [ ]:
# Task B — your code here

**Solution B**

In [ ]:
arms = []
for name, w in [("v1-only", 0.0), ("v2-only", 1.0)]:
    serve.run(ray_serve_app.build_app(w, 2, 2), route_prefix="/", name="default"); time.sleep(3)
    load_test.run(URL, "rest", 4, 2, recs, 1)
    r = load_test.run(URL, "rest", 4, 6, recs, 1); arms.append({"arm": name, "rps": r["rps"], "p95_ms": r["p95_ms"]})
arms = pd.DataFrame(arms); arms["meets_SLO(<=1.5x v1)"] = arms.p95_ms <= 1.5 * arms.p95_ms.iloc[0]
arms

If v2 misses the SLO at this concurrency, then *higher accuracy is not free*: you must add replicas (cost), reduce the forest size, or accept a smaller canary share. That is exactly the trade-off canary analysis exists to expose.

In [ ]:
serve.shutdown(); ray.shutdown(); print("Ray stopped")

### Recap
* `@serve.deployment` = replicas + resources + back-pressure; **fractional CPUs** make laptops viable.
* Application graph: ingress router → model deployments; a **weighted split** is a few lines of Python.
* Manual scaling is bounded by real cores; **autoscaling** follows ongoing requests (scale up fast, down slowly).
* Replicas are supervised: kill one and the controller replaces it.
* Promote/rollback = redeploy the router with a new weight.

**Next:** Lecture 4d — the same ideas (replicas, health checks, rollout, autoscaling) as Kubernetes primitives on a multi-node minikube.